# 01_limpieza.py

Tarea 1 — Limpieza, homogeneización y manejo de missings
Propiedata · Prueba Técnica DS

In [ ]:
import re
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

In [ ]:
# ─────────────────────────────────────────────
# 0. CARGA DE DATOS

In [ ]:
# ─────────────────────────────────────────────
DATA_DIR = "data/"

a_raw = pd.read_csv(DATA_DIR + "listings_plataforma_A.csv")
b_raw = pd.read_csv(DATA_DIR + "listings_plataforma_B.csv")
c_raw = pd.read_csv(DATA_DIR + "listings_plataforma_C.csv")

print(f"Plataforma A: {len(a_raw)} filas, {a_raw.shape[1]} columnas")
print(f"Plataforma B: {len(b_raw)} filas, {b_raw.shape[1]} columnas")
print(f"Plataforma C: {len(c_raw)} filas, {c_raw.shape[1]} columnas")

In [ ]:
# ─────────────────────────────────────────────
# 1. SCHEMA TARGET UNIFICADO

In [ ]:
# ─────────────────────────────────────────────
# Columnas finales del dataset unificado:
# id_original, fuente, tipo_inmueble, barrio, precio_ars,
# superficie_cubierta_m2, superficie_total_m2, ambientes,
# dormitorios, banios, antiguedad_anios, cochera, balcon, pileta,
# latitud, longitud, fecha_publicacion

In [ ]:
# ─────────────────────────────────────────────
# 2. HELPERS GENERALES

In [ ]:
# ─────────────────────────────────────────────

def normalize_barrio(s):
    if pd.isna(s):
        return np.nan
    return s.strip().title()

def normalize_tipo(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip().lower()
    if s in ["departamento", "depto", "dpto"]:
        return "Departamento"
    elif s in ["ph"]:
        return "PH"
    elif s in ["casa"]:
        return "Casa"
    return s.title()

def parse_bool(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, bool):
        return int(v)
    s = str(v).strip().lower()
    if s in ["sí", "si", "true", "1", "yes"]:
        return 1
    elif s in ["no", "false", "0"]:
        return 0
    return np.nan

def parse_date(s, fmt=None):
    try:
        return pd.to_datetime(s, dayfirst=True)
    except Exception:
        return pd.NaT

In [ ]:
# ─────────────────────────────────────────────
# 3. LIMPIEZA PLATAFORMA A

In [ ]:
# ─────────────────────────────────────────────

def clean_precio_a(p):
    """
    Precios en A vienen como '1.267.000' (puntos como miles), o 'USD 741'.
    Descartamos USD (moneda extranjera, no comparables sin tipo de cambio).
    También descartamos valores claramente erróneos (< 100k o > 20M).
    """
    p = str(p).strip()
    if re.search(r'USD|usd|dolar', p, re.IGNORECASE):
        return np.nan
    p_clean = re.sub(r'[^\d]', '', p)
    if not p_clean:
        return np.nan
    val = float(p_clean)
    if val < 100_000 or val > 20_000_000:
        return np.nan
    return val

def clean_superficie_a(s):
    """Viene como '35 m²' o '75' — extraemos el primer número."""
    if pd.isna(s):
        return np.nan
    # Manejo especial de rangos tipo "58 a 68 m2"
    m = re.search(r'(\d+)\s*(?:a|-)\s*\d+', str(s))
    if m:
        return float(m.group(1))   # tomar el mínimo del rango
    m = re.search(r'(\d+\.?\d*)', str(s))
    if m:
        return float(m.group(1))
    return np.nan

def clean_ambientes_a(s):
    """'4 amb.' → 4, '2' → 2"""
    if pd.isna(s):
        return np.nan
    m = re.search(r'(\d+)', str(s))
    return float(m.group(1)) if m else np.nan

def clean_antiguedad_a(s):
    """'4 años' → 4, 'A estrenar' → 0, '23' → 23"""
    if pd.isna(s):
        return np.nan
    s = str(s).strip().lower()
    if 'estrenar' in s:
        return 0.0
    m = re.search(r'(\d+)', s)
    return float(m.group(1)) if m else np.nan

def clean_lat_lon_a(s):
    """Latitudes con coma decimal en lugar de punto: '-34,62...' → -34.62..."""
    if pd.isna(s):
        return np.nan
    s = str(s).strip().replace(',', '.')
    try:
        val = float(s)
        # Coordenadas válidas para Buenos Aires: lat ~ -34.x, lon ~ -58.x
        return val
    except Exception:
        return np.nan

a = a_raw.copy()

a["precio_ars"]              = a["precio"].apply(clean_precio_a)
a["superficie_cubierta_m2"]  = a["superficie_cubierta"].apply(lambda x: x if pd.notna(x) else np.nan)
a["superficie_total_m2"]     = a["superficie_total"].apply(clean_superficie_a)
a["ambientes"]               = a["ambientes"].apply(clean_ambientes_a)
a["dormitorios"]             = a["dormitorios"]
a["banios"]                  = a["banios"]
a["antiguedad_anios"]        = a["antiguedad"].apply(clean_antiguedad_a)
a["cochera"]                 = a["cochera"].apply(parse_bool)
a["balcon"]                  = a["balcon"].apply(parse_bool)
a["pileta"]                  = a["amenities"].apply(
    lambda x: 1 if pd.notna(x) and "pileta" in str(x).lower() else (0 if pd.notna(x) else np.nan)
)
a["latitud"]                 = a["latitud"].apply(clean_lat_lon_a)
a["longitud"]                = a["longitud"].apply(clean_lat_lon_a)
a["barrio"]                  = a["zona_barrio"].apply(normalize_barrio)
a["tipo_inmueble"]           = a["tipo"].apply(normalize_tipo)
a["fecha_publicacion"]       = a["fecha_publicacion"].apply(parse_date)
a["id_original"]             = a["id_aviso"]
a["fuente"]                  = "A"

COLS = ["id_original","fuente","tipo_inmueble","barrio","precio_ars",
        "superficie_cubierta_m2","superficie_total_m2","ambientes",
        "dormitorios","banios","antiguedad_anios","cochera","balcon","pileta",
        "latitud","longitud","fecha_publicacion"]

a_clean = a[COLS].copy()
print(f"\n[A] Filas originales: {len(a_raw)} → limpias: {len(a_clean)}")
print(f"[A] Precio nulo (USD o extremos): {a_clean['precio_ars'].isna().sum()}")

In [ ]:
# ─────────────────────────────────────────────
# 4. LIMPIEZA PLATAFORMA B

In [ ]:
# ─────────────────────────────────────────────
# B es la más limpia. Tiene missings sistemáticos en total_area y age_years.

b = b_raw.copy()

b["precio_ars"]             = b["price_ars"].apply(
    lambda x: x if (pd.notna(x) and 100_000 < x < 20_000_000) else np.nan
)
b["superficie_cubierta_m2"] = b["covered_area_m2"]
b["superficie_total_m2"]    = b["total_area_m2"]
b["ambientes"]              = b["rooms"]
b["dormitorios"]            = b["bedrooms"]
b["banios"]                 = b["bathrooms"]
b["antiguedad_anios"]       = b["age_years"]
b["cochera"]                = b["has_garage"].apply(parse_bool)
b["balcon"]                 = b["has_balcony"].apply(parse_bool)
b["pileta"]                 = b["has_pool"].apply(parse_bool)
b["latitud"]                = b["lat"]
b["longitud"]               = b["lng"]
b["barrio"]                 = b["neighborhood"].apply(normalize_barrio)
b["tipo_inmueble"]          = b["property_type"].apply(normalize_tipo)
b["fecha_publicacion"]      = b["created_at"].apply(parse_date)
b["id_original"]            = b["listing_id"]
b["fuente"]                 = "B"

b_clean = b[COLS].copy()
print(f"\n[B] Filas originales: {len(b_raw)} → limpias: {len(b_clean)}")
print(f"[B] Precio nulo: {b_clean['precio_ars'].isna().sum()}")

In [ ]:
# ─────────────────────────────────────────────
# 5. LIMPIEZA PLATAFORMA C

In [ ]:
# ─────────────────────────────────────────────
# C tiene toda la info estructurada en un campo 'descripcion' de texto libre.
# Hay que parsear ambientes, dormitorios, baños, cochera, balcón, pileta, antigüedad.

def parse_precio_c(p):
    """'$678.000/mes' o 'Consultar'"""
    if pd.isna(p) or str(p).strip().lower() == "consultar":
        return np.nan
    digits = re.sub(r'[^\d]', '', str(p))
    if not digits:
        return np.nan
    val = float(digits)
    if val < 100_000 or val > 20_000_000:
        return np.nan
    return val

def parse_superficie_c(s):
    """'55', '58 a 68 m2' → tomar primer número"""
    if pd.isna(s):
        return np.nan
    m = re.search(r'(\d+)', str(s))
    return float(m.group(1)) if m else np.nan

def parse_coordenadas_c(s):
    """'-34.57599,-58.47863' → (lat, lon)"""
    if pd.isna(s):
        return np.nan, np.nan
    s = str(s).strip().strip('"')
    parts = s.split(',')
    if len(parts) == 2:
        try:
            return float(parts[0]), float(parts[1])
        except Exception:
            return np.nan, np.nan
    return np.nan, np.nan

def parse_desc_field(desc, pattern, default=np.nan):
    """Extrae un número de la descripción según patrón regex."""
    if pd.isna(desc):
        return default
    m = re.search(pattern, str(desc), re.IGNORECASE)
    return float(m.group(1)) if m else default

def parse_desc_bool(desc, keyword):
    """Busca keyword en descripción → 0/1."""
    if pd.isna(desc):
        return np.nan
    return 1 if keyword.lower() in str(desc).lower() else 0

def parse_antiguedad_c(desc):
    """'11 años' → 11, 'a estrenar' → 0"""
    if pd.isna(desc):
        return np.nan
    desc = str(desc).lower()
    if 'estrenar' in desc:
        return 0.0
    m = re.search(r'(\d+)\s*a[ñn]', desc)
    if m:
        return float(m.group(1))
    return np.nan

def parse_date_c(s):
    """'10-03-2024' → date"""
    try:
        return pd.to_datetime(s, dayfirst=True)
    except Exception:
        return pd.NaT

c = c_raw.copy()

coords = c["coordenadas"].apply(parse_coordenadas_c)
c["latitud"]  = [x[0] for x in coords]
c["longitud"] = [x[1] for x in coords]

c["precio_ars"]             = c["precio_publicado"].apply(parse_precio_c)
c["superficie_cubierta_m2"] = c["superficie"].apply(parse_superficie_c)
c["superficie_total_m2"]    = np.nan   # no disponible en C
c["ambientes"]              = c["descripcion"].apply(lambda d: parse_desc_field(d, r'(\d+)\s*amb'))
c["dormitorios"]            = c["descripcion"].apply(lambda d: parse_desc_field(d, r'(\d+)\s*dorm'))
c["banios"]                 = c["descripcion"].apply(lambda d: parse_desc_field(d, r'(\d+)\s*ba[ñn]'))
c["antiguedad_anios"]       = c["descripcion"].apply(parse_antiguedad_c)
c["cochera"]                = c["descripcion"].apply(lambda d: parse_desc_bool(d, "cochera"))
c["balcon"]                 = c["descripcion"].apply(lambda d: parse_desc_bool(d, "balcón"))
c["pileta"]                 = c["descripcion"].apply(lambda d: parse_desc_bool(d, "pileta"))
c["barrio"]                 = c["barrio"].apply(normalize_barrio)
c["tipo_inmueble"]          = c["tipo_inmueble"].apply(normalize_tipo)
c["fecha_publicacion"]      = c["publicado"].apply(parse_date_c)
c["id_original"]            = c["codigo"]
c["fuente"]                 = "C"

c_clean = c[COLS].copy()
print(f"\n[C] Filas originales: {len(c_raw)} → limpias: {len(c_clean)}")
print(f"[C] Precio nulo (Consultar o extremos): {c_clean['precio_ars'].isna().sum()}")

In [ ]:
# ─────────────────────────────────────────────
# 6. UNIFICACIÓN

In [ ]:
# ─────────────────────────────────────────────

df = pd.concat([a_clean, b_clean, c_clean], ignore_index=True)
print(f"\nDataset unificado (pre-filtros): {len(df)} filas")

In [ ]:
# ─────────────────────────────────────────────
# 7. FILTROS POST-UNIFICACIÓN

In [ ]:
# ─────────────────────────────────────────────

# 7.1 Descartar filas sin precio (no podemos modelar sin target)
n_antes = len(df)
df = df[df["precio_ars"].notna()].copy()
print(f"Sin precio descartados: {n_antes - len(df)}")

# 7.2 Coordenadas: validar rango esperado para CABA/GBA
#     lat: -34.9 a -34.4 | lon: -58.9 a -58.3
def valid_coords(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return False
    return (-34.9 < lat < -34.3) and (-59.0 < lon < -58.2)

df["coords_validas"] = df.apply(lambda r: valid_coords(r["latitud"], r["longitud"]), axis=1)
print(f"Coordenadas fuera de rango/nulas: {(~df['coords_validas']).sum()}")
# Las mantenemos en el dataset pero marcamos — útiles para features no espaciales
df.loc[~df["coords_validas"], ["latitud","longitud"]] = np.nan
df.drop(columns="coords_validas", inplace=True)

# 7.3 Superficie cubierta: outliers con IQR
q1 = df["superficie_cubierta_m2"].quantile(0.01)
q99 = df["superficie_cubierta_m2"].quantile(0.99)
mask_sup = df["superficie_cubierta_m2"].notna() & (
    (df["superficie_cubierta_m2"] < q1) | (df["superficie_cubierta_m2"] > q99)
)
df.loc[mask_sup, "superficie_cubierta_m2"] = np.nan
print(f"Superficie cubierta fuera de p1-p99 ({q1:.0f}-{q99:.0f} m²): {mask_sup.sum()} → NaN")

# 7.4 Precio: outliers con IQR (ya filtramos < 100k y > 20M, segunda pasada)
q05 = df["precio_ars"].quantile(0.005)
q995 = df["precio_ars"].quantile(0.995)
mask_precio = (df["precio_ars"] < q05) | (df["precio_ars"] > q995)
df.loc[mask_precio, "precio_ars"] = np.nan
df = df[df["precio_ars"].notna()].copy()
print(f"Precio fuera de p0.5-p99.5 ({q05:.0f}-{q995:.0f}): {mask_precio.sum()}")

In [ ]:
# ─────────────────────────────────────────────
# 8. MANEJO DE MISSINGS

In [ ]:
# ─────────────────────────────────────────────
# Estrategia por campo:
#
# - superficie_cubierta_m2: imputar mediana por (tipo_inmueble, barrio) —
#   superficie es clave para precio, pero descartar el 25% con nulos
#   sería una pérdida enorme. La mediana por segmento es robusta a outliers.
#
# - superficie_total_m2: C no la tiene. Imputar = superficie_cubierta cuando
#   el 75% de los casos son iguales. Aceptamos esta aproximación.
#
# - ambientes: imputar mediana por (tipo_inmueble, barrio). Variable ordinal,
#   la mediana tiene sentido.
#
# - dormitorios: imputar mediana por (tipo_inmueble, ambientes).
#
# - banios: imputar 1 cuando es nulo si ambientes <= 2, sino mediana.
#
# - antiguedad_anios: alta tasa de missing, especialmente en B (~29%).
#   Imputamos mediana + flag binario 'antiguedad_missing' para que el modelo
#   pueda aprender que "no saber la edad" es informativo.
#
# - cochera/balcon/pileta: son 0/1. NaN en A cuando amenities era nulo
#   — lo tratamos como 'no mencionado' → 0 (conservador).
#
# - latitud/longitud: no imputar. El modelo usará features sin coords cuando
#   estén ausentes (ver notebook 02).

# 8.1 Flag de antiguedad missing antes de imputar
df["antiguedad_missing"] = df["antiguedad_anios"].isna().astype(int)

# 8.2 Imputación por grupo
def impute_by_group(df, col, group_cols):
    medians = df.groupby(group_cols)[col].median()
    def fill(row):
        if pd.notna(row[col]):
            return row[col]
        key = tuple(row[g] for g in group_cols)
        return medians.get(key, df[col].median())
    df[col] = df.apply(fill, axis=1)
    return df

df = impute_by_group(df, "superficie_cubierta_m2", ["tipo_inmueble","barrio"])
# Para los que aún quedaron nulos (barrio raro sin datos), usar mediana global
df["superficie_cubierta_m2"].fillna(df["superficie_cubierta_m2"].median(), inplace=True)

# superficie_total ≈ superficie_cubierta si falta
df["superficie_total_m2"] = df["superficie_total_m2"].fillna(df["superficie_cubierta_m2"])

df = impute_by_group(df, "ambientes", ["tipo_inmueble","barrio"])
df["ambientes"].fillna(df["ambientes"].median(), inplace=True)

df = impute_by_group(df, "dormitorios", ["tipo_inmueble","ambientes"])
df["dormitorios"].fillna(df["dormitorios"].median(), inplace=True)

df = impute_by_group(df, "banios", ["tipo_inmueble","ambientes"])
df["banios"].fillna(1.0, inplace=True)

df = impute_by_group(df, "antiguedad_anios", ["tipo_inmueble","barrio"])
df["antiguedad_anios"].fillna(df["antiguedad_anios"].median(), inplace=True)

# 8.3 Cochera/balcon/pileta: NaN → 0
for col in ["cochera","balcon","pileta"]:
    df[col].fillna(0, inplace=True)

# 8.4 Barrio nulo → "Desconocido"
df["barrio"].fillna("Desconocido", inplace=True)

In [ ]:
# ─────────────────────────────────────────────
# 9. FEATURES ADICIONALES

In [ ]:
# ─────────────────────────────────────────────

# Año y mes de publicación (útil para capturar inflación)
df["anio_publicacion"] = df["fecha_publicacion"].dt.year
df["mes_publicacion"]  = df["fecha_publicacion"].dt.month

# Log del precio (target transformado — más simétrico)
df["log_precio"] = np.log1p(df["precio_ars"])

# Precio por m²
df["precio_por_m2"] = df["precio_ars"] / df["superficie_cubierta_m2"]

print(f"\nDataset final limpio: {len(df)} filas, {df.shape[1]} columnas")
print("\nMissings finales:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# ─────────────────────────────────────────────
# 10. GUARDAR OUTPUT

In [ ]:
# ─────────────────────────────────────────────

import os
os.makedirs("output", exist_ok=True)
df.to_parquet("output/dataset_unificado.parquet", index=False)
print("\n✓ Guardado en output/dataset_unificado.parquet")

In [ ]:
# ─────────────────────────────────────────────
# RESUMEN DE DECISIONES

In [ ]:
# ─────────────────────────────────────────────
print("""
══════════════════════════════════════════════════════
RESUMEN DE DECISIONES CLAVE
══════════════════════════════════════════════════════

SCHEMA TARGET
─────────────
Se definieron 17 columnas comunes más flags derivados.
La superficie_total de C (inexistente) se aproxima con
la cubierta, asumiendo que en departamentos suelen coincidir.

LIMPIEZA POR PLATAFORMA
────────────────────────
A: Precios en USD descartados (moneda diferente, sin TC confiable).
   Precios con formato '1.267.000' (puntos como separadores de miles) normalizados.
   Latitudes con coma decimal ('−34,62...') corregidas a punto.
   Superficie y ambientes venían con texto ('35 m²', '4 amb.') → regex.
   Antigüedad 'A estrenar' → 0 años.

B: La más limpia. Se aplicaron solo filtros de rango y normalización de tipos.
   age_years tiene ~29% de nulos → flag + imputación por grupo.

C: Toda la información estructural estaba en el campo descripción libre.
   Se parseó con regex: ambientes, dormitorios, baños, cochera, balcón, pileta,
   antigüedad. La columna superficie_total_m2 no existe en C.
   Precios 'Consultar' → NaN (descartados, no modelables).

OUTLIERS
────────
Precio: se aplicaron dos filtros:
  1) Reglas de dominio: < 100k o > 20M → NaN (errores de carga).
  2) IQR p0.5 - p99.5 para eliminar colas extremas residuales.
USD no se convirtió por falta de fecha de tipo de cambio confiable.

Superficie cubierta: p1-p99 por plataforma unificada.

MISSINGS
────────
- superficie_cubierta_m2: imputación mediana por (tipo_inmueble, barrio).
  Descartarlos implicaría perder ~15% del dataset sin ganar calidad.
- antiguedad_anios: imputación + flag binario 'antiguedad_missing'.
  El flag permite al modelo diferenciar "propiedad nueva" de "dato ausente".
- cochera/balcon/pileta: NaN → 0 (conservador: si no se menciona, se asume ausente).
- latitud/longitud: NO se imputan — usarlas incorrectamente distorsionaría
  features geoespaciales más que omitirlas.

LIMITACIONES
────────────
- No se resolvió el matching entre plataformas (ver PROPUESTA.md §4.1).
  El dataset puede contener duplicados del mismo inmueble.
- La conversión USD→ARS no se realizó: requeriría una serie histórica
  de tipo de cambio alineada a fecha_publicacion.
- La imputación de descripción en C asume patrones de texto consistentes;
  descripciones no estándar generan NaN adicionales.
══════════════════════════════════════════════════════
""")